# BrainIAC (Tak 2026) — SIMON batch

**Before running:** Runtime → Change runtime type → **GPU (L4 or T4)**

Runs the BrainIAC foundation model (ViT-B, SimCLR pretraining) for brain age prediction on SIMON T1 scans.  
Saves results to Google Drive in the same format as `synthba_predictions.csv`.

**Paper:** Tak et al. (2026). A generalizable foundation model for analysis of human brain MRI.  
*Nature Neuroscience*. https://doi.org/10.1038/s41593-026-02202-6  
**Repo:** https://github.com/AIM-KannLab/BrainIAC

### Technical notes
- Cell 4 downloads weights automatically from Dropbox (~1.3 GB total) — no manual setup needed
- BrainIAC requires **preprocessed** T1w: N4 bias correction + skull stripping + MNI registration (Cell 7)
- Images resized to 96×96×96 internally by BrainIAC
- Input CSV columns: `pat_id`, `label` (age **in months**)
- Output column: `predicted_value` (in months) — converted back to years in Cell 10

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone repos
import os
import subprocess

REPO = '/content/faceage-to-brainage'
BRAINIAC_REPO = '/content/BrainIAC'

if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/kondratevakate/faceage-to-brainage.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--rebase'], check=False)

if not os.path.exists(BRAINIAC_REPO):
    subprocess.run(['git', 'clone', 'https://github.com/AIM-KannLab/BrainIAC.git', BRAINIAC_REPO], check=True)
else:
    subprocess.run(['git', '-C', BRAINIAC_REPO, 'pull', '--rebase'], check=False)

print('Repos ready')
print('BrainIAC src contents:', os.listdir(f'{BRAINIAC_REPO}/src'))

In [ ]:
import subprocess, sys
from pathlib import Path

VENV = Path("/content/brainiac_env")

def run(cmd, check=True):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-3000:])
        if check:
            raise RuntimeError(f"Command failed: {cmd}")
    return r

if not VENV.exists():
    r = run([sys.executable, "-m", "venv", str(VENV)], check=False)
    if r.returncode != 0:
        print("venv failed, fallback to virtualenv")
        run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
        run([sys.executable, "-m", "virtualenv", str(VENV)])

py = str(VENV / "bin" / "python")
print("Python:", py)

In [ ]:
# 3. Install BrainIAC dependencies
#
# BrainIAC requirements.txt pins scipy==1.10.1 which fails on Python 3.12.
# We relax exact pins for packages that have Py3.12-incompatible versions.
# Preprocessing deps (antspyx, deepbet, SimpleITK) are installed in Cell 7.

# 3. Install dependencies
import subprocess
import sys

pkgs = [
    "pytorch-lightning==2.3.3",
    "monai==1.3.2",
    "nibabel==5.2.1",
    "scikit-image==0.21.0",
    "scikit-learn==1.2.2",
    "scipy>=1.11,<1.18",
    "seaborn==0.12.2",
    "numpy>=1.23,<2.0",
    "autograd==1.7.0",
    "matplotlib==3.7.1",
    "SimpleITK==2.4.0",
    "tqdm",
    "pydicom",
    "wandb",
    "lifelines",
    "opencv-python",
    "einops",
    "antspyx",
    "deepbet",
]

r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
    capture_output=True,
    text=True,
)

if r.returncode != 0:
    raise RuntimeError("BrainIAC install failed:\n" + r.stderr[-8000:])

print("BrainIAC dependencies installed")
print("Restarting runtime to avoid numpy/scipy binary mismatch...")

In [ ]:
# 4. Download BrainIAC weights from Dropbox
#
# Verified file sizes (from Dropbox, April 2026):
#   brainage.ckpt  — 0.99 GB  — brain age downstream head
#   BrainIAC.ckpt  — 345 MB   — foundation model (SimCLR ViT-B)
#
# Skip download if file already exists and has the right size (safe to re-run).
# If the Dropbox links expire, get new ones from:
#   https://www.dropbox.com/scl/fo/i51xt63roognvt7vuslbl/AG99uZljziHss5zJz4HiFis?rlkey=9w55le6tslwxlfz6c0viylmjb

import subprocess
from pathlib import Path

BRAINIAC_REPO = '/content/BrainIAC'
LOCAL_CKPT_DIR = Path(BRAINIAC_REPO) / 'src' / 'checkpoints'
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

WEIGHTS = {
    'brainage.ckpt': {
        'url': 'https://www.dropbox.com/scl/fi/ssn3sopefnn0r3x2trf7b/brainage.ckpt?rlkey=r97d57asat9w6tba93p08co3a&st=2jsdgc93&dl=1',
        'min_mb': 900,   # actual: ~990 MB
    },
    'BrainIAC.ckpt': {
        'url': 'https://www.dropbox.com/scl/fi/rjvl9sqqp6d35347wiqo8/BrainIAC.ckpt?rlkey=gr3kof7w8lovh3d0ubrntb64g&st=iruzulj1&dl=1',
        'min_mb': 300,   # actual: ~345 MB
    },
}

for fname, cfg in WEIGHTS.items():
    dest = LOCAL_CKPT_DIR / fname
    if dest.exists() and dest.stat().st_size > cfg['min_mb'] * 1024 * 1024:
        print(f'Already downloaded: {fname} ({dest.stat().st_size / 1e6:.0f} MB)')
        continue
    print(f'Downloading {fname}...')
    r = subprocess.run(
        ['wget', '-q', '--show-progress', '-O', str(dest), cfg['url']],
        check=False,
    )
    if r.returncode != 0 or not dest.exists():
        raise RuntimeError(
            f'Download failed for {fname} (exit code {r.returncode}). '
            'Check network or update the URL in WEIGHTS above.'
        )
    size_mb = dest.stat().st_size / 1e6
    if size_mb < cfg['min_mb']:
        dest.unlink()
        raise RuntimeError(
            f'{fname} downloaded but too small ({size_mb:.0f} MB < {cfg["min_mb"]} MB). '
            'The Dropbox link may have expired — get a fresh link from the shared folder.'
        )
    print(f'  OK: {fname} ({size_mb:.0f} MB)')

BRAINAGE_CKPT = LOCAL_CKPT_DIR / 'brainage.ckpt'
print(f'\nFoundation : {LOCAL_CKPT_DIR / "BrainIAC.ckpt"}')
print(f'Brain age  : {BRAINAGE_CKPT}')

In [ ]:
# 5. GPU check
import subprocess
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

subprocess.run(['nvidia-smi'], check=False)

In [ ]:
# 6. Locate SIMON manifest
from pathlib import Path
import pandas as pd

DRIVE = '/content/drive/MyDrive'

SIMON_ROOT_CANDIDATES = [
    Path(DRIVE) / 'brain_mri' / 't1_only' / 'simon',
    Path(DRIVE) / 'brain_mri' / 'simon',
]
SIMON_ROOT = next((p for p in SIMON_ROOT_CANDIDATES if (p / 'manifest.csv').exists()), None)
if SIMON_ROOT is None:
    print('Searching via rglob (may take 30-60s)...')
    matches = [p.parent for p in Path(DRIVE).rglob('manifest.csv') if p.parent.name.lower() == 'simon']
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one simon/manifest.csv, found {len(matches)}: {matches[:5]}')
    SIMON_ROOT = matches[0]

IMAGES_DIR = SIMON_ROOT / 'images'
DRIVE_OUT = Path(DRIVE) / 'brain_mri' / 'brainage_results' / 'simon' / 'brainiac_predictions.csv'
DRIVE_OUT.parent.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(SIMON_ROOT / 'manifest.csv')

def resolve_t1_path(row):
    candidates = []
    if 't1_path' in row.index and pd.notna(row.get('t1_path')):
        raw = str(row['t1_path']).strip()
        candidates += [Path(raw), IMAGES_DIR / Path(raw).name]
    if 't1_filename' in row.index and pd.notna(row.get('t1_filename')):
        candidates.append(IMAGES_DIR / str(row['t1_filename']).strip())
    for c in candidates:
        if c.exists():
            return str(c)
    return str(candidates[0]) if candidates else ''

manifest['t1_path'] = manifest.apply(resolve_t1_path, axis=1)
missing = [p for p in manifest['t1_path'] if not Path(p).exists()]
if missing:
    raise FileNotFoundError(f'Missing {len(missing)} inputs. First 5: {missing[:5]}')

print('SIMON_ROOT:', SIMON_ROOT)
print('Manifest rows:', len(manifest))
display_cols = [c for c in ['subject_id', 'session_id', 'run', 'acquisition_label', 't1_path', 'age'] if c in manifest.columns]
print(manifest[display_cols].head(3).to_string())

In [ ]:
# 7. Preprocess SIMON scans (N4 + skull strip + MNI registration)
#
# Install preprocessing dependencies here (not in Cell 3) to avoid
# waiting for antspyx compilation during initial setup.
# antspyx may take 5-15 min to install from source on Colab.

import subprocess, sys
print('Installing preprocessing dependencies (antspyx may take ~10 min)...')
for pkg in ['nibabel', 'SimpleITK', 'antspyx', 'deepbet']:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    status = 'ok' if r.returncode == 0 else f'FAILED: {r.stderr[-200:]}'
    print(f'  {pkg}: {status}')

#
# BrainIAC requires preprocessed T1w (bias corrected, skull stripped, MNI space)
# We reuse prepare_sfcn_input() from src/brain_age.py which implements the full pipeline.
#
# Preprocessing output convention:
#   Each scan gets a simple numeric ID: scan_000, scan_001, ...
#   Preprocessed file: /tmp/brainiac_preproc/{pat_id}.nii.gz
#   This avoids illegal characters in filenames from scan_key.

import gc
import sys
import pandas as pd
from pathlib import Path
from tqdm import tqdm

REPO = '/content/faceage-to-brainage'
sys.path.insert(0, REPO)
from src.brain_age import prepare_sfcn_input  # N4 + deepbet + ANTs MNI

PREPROC_DIR = Path('/tmp/brainiac_preproc')
PREPROC_DIR.mkdir(parents=True, exist_ok=True)

SKIP_IF_EXISTS = True  # set False to reprocess everything

# pat_id → scan_key and scan_key → pat_id maps (for joining results back)
patid_to_scankey = {}  # 'scan_042' → full scan_key string
patid_to_chron_age = {}  # 'scan_042' → chron_age in years

def clean_key_part(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ''
    t = str(v).strip()
    return '' if t.lower() == 'nan' else t

def build_scan_key(row):
    parts = [
        clean_key_part(row.get('subject_id')),
        clean_key_part(row.get('session_id')),
        clean_key_part(row.get('run')),
        clean_key_part(row.get('acquisition_label')),
        Path(str(row['t1_path'])).name,
    ]
    return '|'.join([p for p in parts if p])

failed_preproc = []  # populated below; also used in cells 8 and 9

for i, (_, row) in enumerate(tqdm(manifest.iterrows(), total=len(manifest), desc='Preprocessing')):
    pat_id = f'scan_{i:04d}'
    out_path = PREPROC_DIR / f'{pat_id}.nii.gz'
    scan_key = build_scan_key(row)
    chron_age = float(row['age']) if 'age' in row.index and not pd.isna(row.get('age')) else float('nan')

    patid_to_scankey[pat_id] = scan_key
    patid_to_chron_age[pat_id] = chron_age

    if SKIP_IF_EXISTS and out_path.exists():
        continue

    try:
        # Correct function signature (verified from src/brain_age.py:319-327):
        #   prepare_sfcn_input(nifti_path, out_path, skullstrip, skullstrip_command,
        #                      keep_skullstripped, n4_correct, register_mni) -> Path
        # Returns a Path to the saved file; file is already written inside the function.
        result_path = prepare_sfcn_input(
            str(row['t1_path']),
            str(out_path),
            skullstrip=True,
            n4_correct=True,
            register_mni=True,
        )
        if not Path(result_path).exists():
            raise RuntimeError(f'prepare_sfcn_input returned {result_path} but file does not exist')
    except Exception as e:
        print(f'[FAIL] {scan_key}: {e}')
        failed_preproc.append({'pat_id': pat_id, 'scan_key': scan_key, 'error': str(e)})

    gc.collect()

# Count completed
done_preproc = [p for p in PREPROC_DIR.glob('scan_*.nii.gz')]
print(f'\nPreprocessed files: {len(done_preproc)}/{len(manifest)}')
if failed_preproc:
    print(f'Failed: {len(failed_preproc)}')
    for f in failed_preproc[:5]:
        print(f'  {f["scan_key"]}: {f["error"][:100]}')

In [ ]:
# 8. Patch BrainIAC test_inference_finetune.py and build input CSV
#
# BrainIAC config is a Python dict inside test_inference_finetune.py (NOT a YAML file).
# We patch the file to set correct paths, then run it.
#
# Verified config keys from BrainIAC source:
#   checkpoint_path, test_csv_path, root_dir, output_csv_path
#   task_type='regression', image_type='single', num_classes=1

import re
import sys
import subprocess
import pandas as pd
from pathlib import Path

# Re-establish context if this cell is run after a restart
BRAINIAC_REPO = '/content/BrainIAC'
PREPROC_DIR = Path('/tmp/brainiac_preproc')

# Rebuild patid maps if cell 7 was skipped
if 'patid_to_scankey' not in dir() or not patid_to_scankey:
    import pandas as pd
    from pathlib import Path

    DRIVE = '/content/drive/MyDrive'
    SIMON_ROOT_CANDIDATES = [
        Path(DRIVE) / 'brain_mri' / 't1_only' / 'simon',
        Path(DRIVE) / 'brain_mri' / 'simon',
    ]
    SIMON_ROOT = next((p for p in SIMON_ROOT_CANDIDATES if (p / 'manifest.csv').exists()), None)
    if SIMON_ROOT is None:
        SIMON_ROOT = [p.parent for p in Path(DRIVE).rglob('manifest.csv') if p.parent.name.lower() == 'simon'][0]

    manifest = pd.read_csv(SIMON_ROOT / 'manifest.csv')
    IMAGES_DIR = SIMON_ROOT / 'images'

    def resolve_t1_path(row):
        candidates = []
        if 't1_path' in row.index and pd.notna(row.get('t1_path')):
            raw = str(row['t1_path']).strip()
            candidates += [Path(raw), IMAGES_DIR / Path(raw).name]
        if 't1_filename' in row.index and pd.notna(row.get('t1_filename')):
            candidates.append(IMAGES_DIR / str(row['t1_filename']).strip())
        for c in candidates:
            if c.exists():
                return str(c)
        return str(candidates[0]) if candidates else ''

    manifest['t1_path'] = manifest.apply(resolve_t1_path, axis=1)

    def clean_key_part(v):
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return ''
        t = str(v).strip()
        return '' if t.lower() == 'nan' else t

    patid_to_scankey = {}
    patid_to_chron_age = {}
    failed_preproc = []

    for i, (_, row) in enumerate(manifest.iterrows()):
        pat_id = f'scan_{i:04d}'
        parts = [
            clean_key_part(row.get('subject_id')), clean_key_part(row.get('session_id')),
            clean_key_part(row.get('run')), clean_key_part(row.get('acquisition_label')),
            Path(str(row['t1_path'])).name,
        ]
        patid_to_scankey[pat_id] = '|'.join([p for p in parts if p])
        patid_to_chron_age[pat_id] = float(row['age']) if 'age' in row.index and not pd.isna(row.get('age')) else float('nan')

    print(f'Rebuilt patid maps for {len(patid_to_scankey)} scans')

# Keep only scans with preprocessed files
done_preproc_ids = {p.stem for p in PREPROC_DIR.glob('scan_*.nii.gz')}
failed_ids = {f['pat_id'] for f in failed_preproc} if 'failed_preproc' in dir() else set()
ready_ids = [pid for pid in patid_to_scankey if pid in done_preproc_ids]

print(f'Ready for inference: {len(ready_ids)}/{len(patid_to_scankey)} scans')
if len(ready_ids) == 0:
    raise RuntimeError('No preprocessed files found. Run cell 7 first.')

# Build BrainIAC input CSV (pat_id, label in months)
BRAINIAC_INPUT_CSV = Path('/tmp/brainiac_input.csv')
BRAINIAC_OUTPUT_CSV = Path('/tmp/brainiac_output.csv')

input_rows = []
for pid in ready_ids:
    chron_age_years = patid_to_chron_age.get(pid, 0.0)
    import math
    age_months = round(chron_age_years * 12) if not math.isnan(chron_age_years) else 0
    input_rows.append({'pat_id': pid, 'label': age_months})

pd.DataFrame(input_rows).to_csv(BRAINIAC_INPUT_CSV, index=False)
print(f'BrainIAC input CSV: {BRAINIAC_INPUT_CSV} ({len(input_rows)} rows)')

# Find brain age checkpoint
LOCAL_CKPT_DIR = Path(BRAINIAC_REPO) / 'src' / 'checkpoints'
brainage_ckpts = sorted(LOCAL_CKPT_DIR.rglob('*.ckpt'),
                        key=lambda p: 'brainage' not in p.name.lower())
brainage_ckpts = [c for c in brainage_ckpts if 'brainage' in c.name.lower() or 'brain_age' in c.name.lower()]
if not brainage_ckpts:
    all_ckpts = list(LOCAL_CKPT_DIR.rglob('*.ckpt'))
    raise FileNotFoundError(
        f'Brain age checkpoint not found. Available: {[c.name for c in all_ckpts]}'
    )
BRAINAGE_CKPT = brainage_ckpts[0]
print(f'Using checkpoint: {BRAINAGE_CKPT}')

# Patch test_inference_finetune.py
# The file contains a DATASETS dict with a 'brainage_task' entry.
# We update it using regex replacement on the relevant keys.
inference_script = Path(BRAINIAC_REPO) / 'src' / 'test_inference_finetune.py'
if not inference_script.exists():
    raise FileNotFoundError(f'BrainIAC inference script not found: {inference_script}')

with open(inference_script, 'r') as f:
    src = f.read()

# Build the replacement brainage_task config block
new_brainage_task = f'''"brainage_task": {{
        "checkpoint_path": "{BRAINAGE_CKPT}",
        "test_csv_path": "{BRAINIAC_INPUT_CSV}",
        "root_dir": "{PREPROC_DIR}",
        "output_csv_path": "{BRAINIAC_OUTPUT_CSV}",
        "task_type": "regression",
        "image_type": "single",
        "num_classes": 1
    }}'''

# Replace the brainage_task block (handles both original and previously-patched versions)
pattern = r'"brainage_task"\s*:\s*\{[^}]*\}'
if not re.search(pattern, src, re.DOTALL):
    raise ValueError(
        'Could not find "brainage_task" dict in test_inference_finetune.py. '
        f'Check that the file at {inference_script} has the expected structure.'
    )

src_patched = re.sub(pattern, new_brainage_task, src, flags=re.DOTALL)

# Also ensure DATASETS_TO_RUN only includes brainage_task
src_patched = re.sub(
    r'DATASETS_TO_RUN\s*=\s*\[[^\]]*\]',
    'DATASETS_TO_RUN = ["brainage_task"]',
    src_patched
)

with open(inference_script, 'w') as f:
    f.write(src_patched)

print(f'Patched {inference_script}')
print('  checkpoint_path:', BRAINAGE_CKPT)
print('  test_csv_path:', BRAINIAC_INPUT_CSV)
print('  root_dir:', PREPROC_DIR)
print('  output_csv_path:', BRAINIAC_OUTPUT_CSV)

In [ ]:
# 9. Run BrainIAC inference
import subprocess
import sys
from pathlib import Path

BRAINIAC_REPO = '/content/BrainIAC'
BRAINIAC_OUTPUT_CSV = Path('/tmp/brainiac_output.csv')

inference_script = Path(BRAINIAC_REPO) / 'src' / 'test_inference_finetune.py'
cmd = [sys.executable, str(inference_script)]
print('Running:', ' '.join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    cwd=BRAINIAC_REPO,
    timeout=3600,
)

if result.stdout:
    print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])
    raise RuntimeError(f'BrainIAC inference failed (exit code {result.returncode})')

print('BrainIAC inference complete')

if not BRAINIAC_OUTPUT_CSV.exists():
    raise FileNotFoundError(
        f'Output file not created: {BRAINIAC_OUTPUT_CSV}\n'
        'Inference script ran successfully but produced no output. Check stderr above.'
    )
print(f'Output file size: {BRAINIAC_OUTPUT_CSV.stat().st_size} bytes')

In [ ]:
# 10. Parse BrainIAC output, join with manifest, save to Drive
#
# Verified output column from BrainIAC source: 'predicted_value' (in months)
# We convert months → years for the results CSV.

import math
import shutil
import pandas as pd
from pathlib import Path

DRIVE = '/content/drive/MyDrive'
DRIVE_OUT = Path(DRIVE) / 'brain_mri' / 'brainage_results' / 'simon' / 'brainiac_predictions.csv'
DRIVE_OUT.parent.mkdir(parents=True, exist_ok=True)
BRAINIAC_OUTPUT_CSV = Path('/tmp/brainiac_output.csv')

brainiac_out = pd.read_csv(BRAINIAC_OUTPUT_CSV)
print('BrainIAC output columns:', list(brainiac_out.columns))
print(brainiac_out.head(3).to_string())

# Verified output column: 'predicted_value' (regression task)
# Find the ID column and prediction column
id_col = next((c for c in brainiac_out.columns if 'pat_id' in c.lower() or c.lower() == 'id'), None)
pred_col = next((c for c in brainiac_out.columns if 'predicted_value' in c.lower() or 'pred' in c.lower()), None)

if id_col is None:
    raise KeyError(f'No ID column found. Columns: {list(brainiac_out.columns)}')
if pred_col is None:
    raise KeyError(f'No predicted value column found. Columns: {list(brainiac_out.columns)}')

print(f'Using: id={id_col!r}, pred={pred_col!r}')

# Predictions are in months → convert to years
pred_map_years = {
    str(row[id_col]): float(row[pred_col]) / 12.0
    for _, row in brainiac_out.iterrows()
}

# Rebuild failed_preproc if not in scope
if 'failed_preproc' not in dir():
    failed_preproc = []
failed_ids = {f['pat_id'] for f in failed_preproc}

# Build final results dataframe
results = []
for pat_id, scan_key in patid_to_scankey.items():
    chron_age = patid_to_chron_age.get(pat_id, float('nan'))
    pred_years = pred_map_years.get(pat_id, float('nan'))
    gap = pred_years - chron_age if not math.isnan(pred_years) and not math.isnan(chron_age) else float('nan')

    if pat_id in failed_ids:
        status = 'error'
        error_msg = next(f['error'] for f in failed_preproc if f['pat_id'] == pat_id)
    else:
        status = 'ok' if not math.isnan(pred_years) else 'error'
        error_msg = '' if not math.isnan(pred_years) else 'no prediction'

    # Parse subject info from scan_key (format: subject_id|session_id|run|acq|filename)
    key_parts = scan_key.split('|')
    results.append({
        'scan_key': scan_key,
        'pat_id': pat_id,
        'subject_id': key_parts[0] if len(key_parts) > 0 else '',
        'session_id': key_parts[1] if len(key_parts) > 1 else '',
        'run': key_parts[2] if len(key_parts) > 2 else '',
        'acquisition_label': key_parts[3] if len(key_parts) > 3 else '',
        'chron_age': chron_age,
        'predicted_age': pred_years,
        'brain_age_gap': gap,
        'model_name': 'BrainIAC_Tak2026',
        'status': status,
        'error': error_msg,
    })

df_results = pd.DataFrame(results)
df_results.to_csv('/tmp/brainiac_simon_results.csv', index=False)
shutil.copy2('/tmp/brainiac_simon_results.csv', str(DRIVE_OUT))

ok = df_results[df_results['status'] == 'ok']
print(f'Total: {len(df_results)}, ok: {len(ok)}, errors: {len(df_results) - len(ok)}')
print(f'Saved → {DRIVE_OUT}')

In [ ]:
# 11. Summary (run after reopening if runtime was disconnected)
import numpy as np
import pandas as pd
from pathlib import Path

DRIVE = '/content/drive/MyDrive'
DRIVE_OUT = Path(DRIVE) / 'brain_mri' / 'brainage_results' / 'simon' / 'brainiac_predictions.csv'

df = pd.read_csv(DRIVE_OUT)
ok = df[df['status'] == 'ok'].copy()

print('rows total:', len(df))
print('rows ok:', len(ok))

if len(ok):
    err = ok['predicted_age'] - ok['chron_age']
    print('MAE:', float(err.abs().mean()))
    print('RMSE:', float(np.sqrt(max((err ** 2).mean(), 0.0))))
    print('Bias (mean gap):', float(err.mean()))
    print('Predicted age std:', float(ok['predicted_age'].std()))
    print(ok[['subject_id', 'session_id', 'run', 'chron_age', 'predicted_age', 'brain_age_gap']].head(10).to_string())